# Tool-Discovery Agent + Biomni Integration

MCP Server (HTTP mode) + Biomni Agent on Colab.

**流程:** 安装依赖 → 启动 MCP Server (HTTP) → 运行 Discovery Pipeline → Biomni 调用发现的工具

In [ ]:
!pip install -q requests beautifulsoup4 pyyaml pandas tabulate python-dateutil langchain-openai biomni

## Step 1: Clone repo & setup

In [ ]:
import os, subprocess, time, sys

REPO_DIR = "/content/scientific_training2_cxy"
if not os.path.exists(REPO_DIR):
    !cd /content && git clone https://github.com/caixiaoyao2025/scientific_training2_cxy.git

os.chdir(REPO_DIR)
os.makedirs("data", exist_ok=True)
print(f"Working in: {os.getcwd()}")

## Step 2: Start MCP Server (HTTP mode)

In [ ]:
env = os.environ.copy()
env["MCP_DATA_ROOT"] = os.path.join(os.getcwd(), "data")
env["MCP_APP_ROOT"] = os.getcwd()
env["MCP_TRANSPORT"] = "streamable-http"
env["MCP_HOST"] = "0.0.0.0"
env["MCP_PORT"] = "8765"
env["MCP_PATH"] = "/mcp"

server_proc = subprocess.Popen(
    [sys.executable, "server.py"],
    env=env, stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True
)
time.sleep(3)
print(f"MCP server PID={server_proc.pid}, returncode={server_proc.poll()}")

## Step 3: Verify MCP Server

In [ ]:
import requests, json

# MCP over HTTP: send JSON-RPC request
def mcp_call(method, params=None):
    payload = {"jsonrpc": "2.0", "id": 1, "method": method, "params": params or {}}
    resp = requests.post("http://127.0.0.1:8765/mcp", json=payload, timeout=10)
    return resp.json()

# Initialize
init_payload = {
    "jsonrpc": "2.0", "id": 0, "method": "initialize",
    "params": {
        "protocolVersion": "2024-11-05",
        "capabilities": {},
        "clientInfo": {"name": "colab-test", "version": "1.0"}
    }
}
resp = requests.post("http://127.0.0.1:8765/mcp", json=init_payload, timeout=10)
print(f"Init status: {resp.status_code}")

# List tools
tools_resp = mcp_call("tools/list")
tools = tools_resp.get("result", {}).get("tools", [])
print(f"\nFound {len(tools)} tools:")
for t in tools:
    print(f"  - {t['name']}: {t.get('description', '')[:60]}")

## Step 4: Run Discovery Pipeline

In [ ]:
from run_pipeline import run_full_pipeline
run_full_pipeline(query="bioinformatics software tool github", max_results=8)

## Step 5: Connect Biomni to MCP Server

In [ ]:
# Kill the standalone server - Biomni will start its own
server_proc.terminate()
server_proc.wait()
print("Standalone server stopped.")

In [ ]:
import yaml

# Write MCP config - stdio mode, Biomni will launch server.py as subprocess
config = {
    "mcp_servers": {
        "bio-mcp": {
            "command": [sys.executable, os.path.join(os.getcwd(), "server.py")],
        }
    }
}
config_path = os.path.join(os.getcwd(), "mcp_config_colab.yaml")
with open(config_path, "w") as f:
    yaml.dump(config, f)
print(f"Config: {config_path}")

In [ ]:
from biomni.agent import A1

agent = A1(
    path=os.path.join(os.getcwd(), "data"),
    llm="Qwen/Qwen2.5-14B-Instruct",
    source="Custom",
    base_url="https://api.siliconflow.cn/v1",
    api_key="sk-lufftravmhzgdpudfvbvzwrlfyctebizytmtlbynyiohtkij",
    expected_data_lake_files=[],
)
agent.add_mcp(config_path=config_path)

result = agent.go("List all available MCP tools using list_registered_tools")
print(result)